# 03 — Attribution graph

**Phase 3** of `vault/01-projects/code/mech-interp-codebase.md`: install/verify circuit-tracer, render an attribution graph for a single chosen prompt, annotate the Phase-2 features in the graph, and sanity-check that the active path makes prose-level sense.

## What this notebook does

1. Verifies the `circuit-tracer` install resolves and that the Gemma-2-2B `ReplacementModel` loads with the bundled Gemma Scope cross-layer transcoders (CLTs / PLTs).
2. Runs `attribute(...)` on the refusal-cued prompt from Phase 1 to produce a raw attribution graph (`.pt`).
3. Prunes the graph (`prune_graph`) to a readable size and writes the graph-files bundle for the visualiser.
4. Cross-references the live nodes against the Phase-2 picks (`../data/phase2_sae_features.pt`) and prints which Phase-2 feature indices survived pruning — those are the nodes to annotate in the local server.
5. Boots the local visualisation server (`circuit_tracer.frontend.local_server.serve`) so the graph can be explored in-browser; the user inspects, groups Phase-2 features into supernodes, and annotates active paths.
6. Documents the Llama-3.2-1B fallback path if the Gemma-2-2B graph turns out to be unreadable — and records the fallback decision into `## Log`.

## Why this matters

Ameisen et al. 2025 (`zotero://select/library/items/ZQIF7Z59`) introduced cross-layer transcoders + attribution graphs as the first method that produces **input-specific causal computational graphs** over interpretable features on a deployed model. The companion paper *On the Biology of a Large Language Model* (Lindsey et al. 2025, `zotero://select/library/items/5XZVJMTF`) walked nine case studies on Claude 3.5 Haiku — multi-hop reasoning, poetry planning, refusals, jailbreaks, chain-of-thought faithfulness, hidden goals. Anthropic open-sourced the `circuit-tracer` library on 2025-05-29; this notebook reproduces the method on Gemma-2-2B, the smallest model the tool supports, so the technique can be driven locally rather than read about.

The Phase-2 features (top-k differential picks on the refusal vs compliance contrast) are *candidates* for the path that produces the refusal-leaning next-token distribution. The attribution graph is where their causal role gets tested: a feature that fires correlationally on the divergent slot but has zero direct effect on the relevant logits is a *prompt-conditioned correlate*, not part of the circuit. Phase 2 generates hypotheses; Phase 3 falsifies them.

## Memory caveat

`circuit-tracer` loads the model **and** the transcoders together (`ReplacementModel.from_pretrained`). With Gemma-2-2B + Gemma Scope CLTs / PLTs this is ~10–12 GB resident plus transient duplicates during attribution. Local CPU is feasible but slow (~minutes per attribution pass). On Apple Silicon, Metal (`mps`) is usable but bfloat16 has known issues in `TransformerLens` 3.x; we follow Phase 1/2 in preferring fp16 on MPS, bf16 on CUDA, fp32 on CPU. If MPS OOMs, fall through to CPU fp32 — slower but reliable.

## Citations

- Attribution-graph method paper: Ameisen et al. 2025 — `zotero://select/library/items/ZQIF7Z59`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art § Crosscoders and transcoders extend SAEs across layers.
- Biology-of-an-LLM case studies: Lindsey et al. 2025 — `zotero://select/library/items/5XZVJMTF`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art.
- Transcoders prior art: Dunefsky, Chlenski, Nanda 2024 — `zotero://select/library/items/WC37AC2M`. See `vault/03-areas/research/mech-interp-overview.md` § Mechanism.
- Sparse Feature Circuits (the predecessor pattern: SAE features as nodes in an automatically-discovered causal graph, with feature-level edits): Marks et al. 2024 — `zotero://select/library/items/5IG9HV6K`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art.
- Towards Automated Circuit Discovery (the methodology lineage circuit-tracer extends): Conmy et al. 2023 — `zotero://select/library/items/VKU2P5AB`. See `vault/03-areas/research/mech-interp-overview.md` § Historical lineage.

## 1. Imports + paths

In [ ]:
from pathlib import Path

import torch

import mech_interp

DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(exist_ok=True)
GRAPH_DIR = DATA_DIR / 'phase3_graphs'
GRAPH_DIR.mkdir(exist_ok=True)
GRAPH_FILES_DIR = DATA_DIR / 'phase3_graph_files'
GRAPH_FILES_DIR.mkdir(exist_ok=True)

torch.set_grad_enabled(False)  # attribute() flips this internally where needed

PHASE2_CHECKPOINT = DATA_DIR / 'phase2_sae_features.pt'
assert PHASE2_CHECKPOINT.exists(), (
    f'expected {PHASE2_CHECKPOINT} from 02_sae_features.ipynb; run that first'
)

## 2. Verify the `circuit-tracer` install

Imports happen first because a stale install (e.g., a git-only version pinned before circuit-tracer landed on PyPI) is the most common breakage. If this cell raises, run `uv sync` from `code/mech-interp-codebase/` and restart the kernel.

In [ ]:
# Ref: Circuit Tracing — zotero://select/library/items/ZQIF7Z59
import importlib.metadata as md
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.graph import prune_graph
from circuit_tracer.utils import create_graph_files

print(f'circuit-tracer : {md.version("circuit-tracer")}')
print(f'torch          : {torch.__version__}')
print(f'cuda available : {torch.cuda.is_available()}')
print(f'mps available  : {torch.backends.mps.is_available()}')

## 3. Load `ReplacementModel` with Gemma Scope transcoders

`transcoder_name='gemma'` is `circuit-tracer`'s shorthand for the Gemma Scope per-layer transcoders (PLTs) hosted on HuggingFace under `mntss/gemma-scope-transcoders`. This loads transcoders for all layers, not just `LAYER` from Phase 2 — attribution traces edges across the full residual stack.

The `ReplacementModel` is a separate object from the `HookedTransformer` loaded by `mech_interp.get_model` in Phase 1/2: it inherits from `HookedTransformer` (with the `transformerlens` backend) but additionally wires in the transcoders. Unfortunately that means we pay the model-load cost again here — circuit-tracer doesn't accept a pre-loaded `HookedTransformer`. To keep total RAM under control, **shut down the kernel(s) for `00`–`02` before running this cell.** The module docstring in `src/mech_interp/__init__.py` flags the same constraint.

If MPS OOMs, set `device='cpu'` explicitly and accept the slowdown (~5–10× slower per attribution pass).

In [ ]:
DEVICE, DTYPE = mech_interp.select_device_dtype()
print(f'device={DEVICE} dtype={DTYPE}')

MODEL_NAME = 'google/gemma-2-2b'
TRANSCODER_NAME = 'gemma'  # shorthand for Gemma Scope transcoders
BACKEND = 'transformerlens'

rmodel = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=DTYPE,
    backend=BACKEND,
)
rmodel.eval()
print(f'n_layers : {rmodel.cfg.n_layers}')
print(f'd_model  : {rmodel.cfg.d_model}')

## 4. Pick the prompt + attribution parameters

We use the refusal prompt from Phase 1/2 so the Phase-2 picks have a chance of appearing as nodes in the graph. Attribution defaults are taken from the `circuit-tracer` README; the only knobs worth tweaking are:

- `max_n_logits` (10) — attribute from the top-10 next-token logits, weighted by their probability share.
- `desired_logit_prob` (0.95) — but stop early if 95% of the next-token probability mass is already covered.
- `max_feature_nodes` — keep this small (~2048–4096) on local hardware to keep RAM in check. The official default is 7500 (Colab GPU) but we're on CPU/MPS.
- `offload` — push parts of the model to CPU (when on MPS/CUDA) or to disk (Colab). Set to `None` on a pure-CPU run.

`verbose=True` lights up the tqdm bar so you can see progress; one pass on Gemma-2-2B is 2–5 minutes on CPU, 30–60s on a GPU.

In [ ]:
phase2 = torch.load(PHASE2_CHECKPOINT, weights_only=False)
PROMPT = phase2['prompts']['refusal']
print(f'prompt: {PROMPT!r}')

MAX_N_LOGITS = 10
DESIRED_LOGIT_PROB = 0.95
MAX_FEATURE_NODES = 4096      # smaller than the 7500 Colab default; ↑ if RAM allows
BATCH_SIZE = 128              # backward-pass batch; halve if OOM
OFFLOAD = 'cpu' if DEVICE != 'cpu' else None  # on pure CPU, no further offload helps
VERBOSE = True

## 5. Run attribution

This is the heavy compute step. On local CPU it's slow but it works; on MPS it's faster but risks OOMing on the backward pass — if it does, drop `MAX_FEATURE_NODES` to 2048 and `BATCH_SIZE` to 64, or move the cell to CPU.

In [ ]:
graph = attribute(
    prompt=PROMPT,
    model=rmodel,
    max_n_logits=MAX_N_LOGITS,
    desired_logit_prob=DESIRED_LOGIT_PROB,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
    offload=OFFLOAD,
    verbose=VERBOSE,
)
print(f'real_features      : {len(graph.real_features)}')
print(f'logit_targets      : {len(graph.logit_targets)}')
print(f'adjacency_matrix   : {tuple(graph.adjacency_matrix.shape)}')

graph_path = GRAPH_DIR / 'refusal_prompt.pt'
graph.to_pt(graph_path)
print(f'saved {graph_path} ({graph_path.stat().st_size / 1e6:.1f} MB)')

## 6. Prune + create graph files

Default thresholds (`node_threshold=0.8`, `edge_threshold=0.98`) preserve enough of the graph to capture 80% of the cumulative direct effect on logits and 98% of the cumulative edge weight. Lower for a more aggressive prune (cleaner, smaller, less complete); raise toward 1.0 for a fuller graph.

`create_graph_files` produces a JSON bundle the local server consumes. The `slug` becomes the URL path (`localhost:<port>/?slug=<slug>`).

In [ ]:
NODE_THRESHOLD = 0.8
EDGE_THRESHOLD = 0.98
SLUG = 'refusal-prompt-gemma2-2b'

create_graph_files(
    graph_or_path=graph_path,
    slug=SLUG,
    output_path=str(GRAPH_FILES_DIR),
    node_threshold=NODE_THRESHOLD,
    edge_threshold=EDGE_THRESHOLD,
)
print(f'wrote graph files under {GRAPH_FILES_DIR}/ (slug={SLUG})')

## 7. Cross-reference live nodes against Phase-2 picks

`graph.real_features` is a list of `(layer, position, feature_idx)` tuples — one per non-zero transcoder feature node in the unpruned graph. We filter to those at the Phase-2 layer and check which of the Phase-2 picks survived attribution. **These are the nodes worth annotating in the local server in §8.**

Caveat: circuit-tracer's transcoders are cross-layer (CLT / PLT — *transcoders*, not autoencoders), so their feature index space is **not** the same as the Gemma Scope width-16k *SAE* feature index space used in Phase 2. The naïve idx-equality check below will likely return zero overlaps. The right comparison is decoder-direction similarity: project the transcoder feature decoder rows onto the SAE decoder columns and find the cosine-nearest match. We do the cheap version (idx-equality) and surface the gap in the printout — implementing the cosine-match is a one-cell follow-up if the gap matters for your write-up.

In [ ]:
phase2_layer = phase2['sae']['layer']
phase2_refusal_picks = {row['feature_idx'] for row in phase2['picks']['refusal_leaning']}
phase2_compliance_picks = {row['feature_idx'] for row in phase2['picks']['compliance_leaning']}

# graph.real_features is (layer, position, feature_idx) tuples on the transcoder side.
live_at_phase2_layer = [
    (layer, pos, fidx)
    for layer, pos, fidx in graph.real_features
    if layer == phase2_layer
]

print(f'phase-2 layer: {phase2_layer}')
print(f'live transcoder feature nodes at that layer: {len(live_at_phase2_layer)}')

overlap_refusal = {(l, p, f) for l, p, f in live_at_phase2_layer
                   if f in phase2_refusal_picks}
overlap_compliance = {(l, p, f) for l, p, f in live_at_phase2_layer
                      if f in phase2_compliance_picks}

print(f'idx-equality overlap with refusal-leaning SAE picks    : {len(overlap_refusal)}')
print(f'idx-equality overlap with compliance-leaning SAE picks : {len(overlap_compliance)}')
print()
print('Note: SAE idx and transcoder idx are different feature spaces.')
print('Zero or near-zero overlap is expected — see notebook §7 prose for the cosine-match follow-up.')

print()
print('Top 10 live transcoder nodes at the Phase-2 layer (by feature_idx):')
for layer, pos, fidx in sorted(live_at_phase2_layer, key=lambda t: t[2])[:10]:
    print(f'  layer={layer} pos={pos} feature_idx={fidx}')

## 8. Boot the local server

The visualiser is the place where this notebook stops being a script and starts being a research tool. Click into the graph in your browser, identify the active path from input tokens → mid-stream features → output logits, and use Ctrl/Cmd-click to pin nodes onto the right-hand subgraph pane. Hold `G` and click pinned nodes to bundle them into supernodes (good for grouping the Phase-2 picks if any survived).

Stop the server with `server.stop()` in the next cell when done.

In [ ]:
from circuit_tracer.frontend.local_server import serve

PORT = 8046
server = serve(data_dir=str(GRAPH_FILES_DIR), port=PORT)

print(f'serving at http://localhost:{PORT}/index.html?slug={SLUG}')
print('(If this is a remote machine, port-forward 8046 to your laptop first.)')

# Embedded IFrame — comment out if your environment can't render it.
from IPython.display import IFrame  # noqa: E402
IFrame(src=f'http://localhost:{PORT}/index.html?slug={SLUG}', width='100%', height='800px')

In [ ]:
# Run this cell when you're done inspecting.
# server.stop()

## 9. Sanity-check the active path makes prose sense

Once the graph is rendered, walk the active path from the divergent input tokens (`' lethal'`, `' toxin'`) to the top output logits, and ask:

1. **Do the mid-stream feature nodes' top-activating prompts (visible on hover / in the right-hand pane) read as a coherent story?** "Harm-topic" → "preamble about danger" → "first-token of cautious continuation" is a story; if the nodes are uncorrelated semantically, the prompt may not have a clean circuit at this scale.
2. **Are any Phase-2 picks pinned in the subgraph?** If yes — write down their annotations in §10. If no — note that in §11 and in `## Log`. (Mismatch between SAE-space and transcoder-space is expected; the question is whether the *story* the transcoder-space tells is consistent with the *story* the SAE-space told in Phase 2.)
3. **What's the top output logit doing?** On a base model the refusal-prompt continuation is unlikely to be `' I'` (the typical refusal lead). It's more likely a topical word (`' Do'`, `' Find'`, etc.). Whatever it is — does the active path produce it from the divergent tokens, or does it look like the scaffold tokens are doing all the work?

## 10. Annotations

After grouping + annotating supernodes in the local server, paste the annotation summary below. Aim for one short paragraph per supernode — what does this cluster of nodes seem to be doing in the circuit? Same evidence standards as Phase 2 §9: token-level vs concept-level, where it fires, whether the Phase-2 gloss agrees.

In [ ]:
# Fill in after running §8. Each entry is one supernode you annotated in the server.
ANNOTATIONS: dict[str, str] = {
    # 'harm-topic features (layer 18-22)': """
    # Cluster of cross-layer transcoder features pinned to the divergent slot
    # tokens ('lethal', 'toxin'). Top-activating prompts in the dashboard read
    # as 'dangerous substance' content. Active path to the top output logit
    # passes through this cluster, so it has nonzero direct effect on the
    # logits — i.e., a candidate causal node, not just a correlate.
    # """,
}

for label, prose in ANNOTATIONS.items():
    print(f'--- {label} ---')
    print(prose.strip())
    print()

## 11. Llama-3.2-1B fallback

If the Gemma-2-2B attribution graph turned out to be unreadable (default thresholds yielded too many nodes, lowering them stripped the active path, or the visualiser hung on the JSON), the spec authorises falling back to Llama-3.2-1B. The mechanics are nearly identical — only the `MODEL_NAME` and `TRANSCODER_NAME` change.

**Document the fallback decision in `## Log`** of `vault/01-projects/code/mech-interp-codebase.md` before running the fallback cell, so the reason for switching is captured.

```python
# Uncomment to run the fallback.
# del rmodel; torch.cuda.empty_cache() if torch.cuda.is_available() else None
# rmodel_llama = ReplacementModel.from_pretrained(
#     'meta-llama/Llama-3.2-1B',
#     'llama',
#     dtype=DTYPE,
#     backend=BACKEND,
# )
# graph_llama = attribute(
#     prompt=PROMPT,
#     model=rmodel_llama,
#     max_n_logits=MAX_N_LOGITS,
#     desired_logit_prob=DESIRED_LOGIT_PROB,
#     batch_size=BATCH_SIZE,
#     max_feature_nodes=MAX_FEATURE_NODES,
#     offload=OFFLOAD,
#     verbose=VERBOSE,
# )
# graph_path_llama = GRAPH_DIR / 'refusal_prompt_llama32_1b.pt'
# graph_llama.to_pt(graph_path_llama)
# create_graph_files(graph_or_path=graph_path_llama, slug='refusal-prompt-llama32-1b',
#                    output_path=str(GRAPH_FILES_DIR),
#                    node_threshold=NODE_THRESHOLD, edge_threshold=EDGE_THRESHOLD)
```

Note: Llama-3.2-1B is also a base model, so the same caveat from Phase 1/2 applies — we're sketching a base-model proxy for behaviour the literature studies on instruction-tuned variants.

## 12. Observations to log

Jot the answers below into `## Log` of `vault/01-projects/code/mech-interp-codebase.md` and into the Phase-3 findings file the orchestrator will append back into the linked research notes.

Prompts:

1. **Did the graph render at the default thresholds (0.8 / 0.98), or did you have to retune?** Record the final `(node_threshold, edge_threshold)` you settled on and one sentence on why.
2. **Did the active path read as a coherent story?** If yes — sketch it in one sentence ("input tokens X → mid-stream cluster Y → logit Z"). If no — what made it incoherent (too many parallel paths, error nodes dominating, transcoders too sparse)?
3. **Did any Phase-2 picks survive into the graph (after the cosine-match follow-up, if you did it)?** If yes — does their annotated role in the attribution graph match the gloss you wrote in Phase 2 §9? If no — write one sentence on what to revise in the Phase-2 gloss.
4. **What's the smallest causal experiment this graph suggests?** Typical answer: "ablate supernode X by zeroing its features and re-run a forward pass; predict the top logit changes from L1 to L2." This becomes Phase 5 if it survives the Phase-4 write-up reflection.

## Open issues to surface to the user

- **Feature-index mismatch between Gemma Scope SAE (Phase 2) and Gemma Scope transcoders (Phase 3).** Same model, *different* dictionaries — the cheap idx-equality check in §7 surfaces this; the proper fix is decoder-cosine matching. Document the gap if you don't implement the fix.
- **Error nodes can dominate the graph on small models.** If a few of the top-by-influence nodes turn out to be `error_*` rather than `feature_*`, the transcoders are failing to reconstruct the residual stream at those positions and the attribution graph is making up for it with error edges. This is a known limitation of the small-model setting noted in the circuit-tracer README — record the count if it dominates.
- **The graph is correlational on a single prompt.** The right validation is intervention: clamp the candidate nodes (the local server's "Steer" button does this) and check that the top-logit distribution shifts as the graph predicts. This notebook surfaces the graph; the intervention is the next worthwhile experiment if the active path looked clean.
- **MPS bf16 quirks.** If you ran into NaNs or garbled feature values on MPS, drop to CPU fp32 — slower but reliable. Record in `## Log` if the fallback was needed.